# Text-to-SVG Inference v3 — Maximum Quality

**Improvements over v2:**
- Aggressive number rounding: `93.30000305175781` → `93.3` (~70% size reduction per number)
- Remove junk `filling="0"` attribute from training data artifacts
- Normalize `0.0 0.0 200.0 200.0` → `0 0 200 200` in viewBox
- Multi-candidate generation: 3 outputs per prompt, best selected by visual content score
- System prompt hints at 200x200 coordinate space (matches 82% of training data)
- Compactness: strip SVG comments, normalize whitespace in attributes

In [ ]:
%pip install -q unsloth==2026.3.8 transformers==5.3.0 peft==0.18.1 \
    bitsandbytes==0.49.2 accelerate==1.13.0 trl==0.24.0 \
    cairosvg lxml

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────────
BASE_MODEL_PATH  = "/kaggle/input/svg-base-model"
LORA_PATH        = "/kaggle/input/svg-lora-model/lora"
TEST_CSV         = "/kaggle/input/svg-competition/test.csv"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"
CHECKPOINT_PATH  = "/kaggle/working/submission_checkpoint.csv"

# Generation config
MAX_NEW_TOKENS   = 3000
NUM_CANDIDATES   = 3      # generate N candidates, pick best
CAND_TEMPS       = [None, 0.3, 0.7]  # None = greedy
SAVE_EVERY       = 50

print("Paths configured.")

In [ ]:
# ── SVG constants & helpers ─────────────────────────────────────────────────────
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clippath", "mask", "lineargradient",
    "radialgradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter",
}
DISALLOWED_TAGS = {
    "script", "animate", "animatetransform", "animatemotion",
    "animatecolor", "set", "foreignobject",
}
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)
SVG_OPEN  = re.compile(r"<svg[\s\S]*", re.IGNORECASE)

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 200 200">'
    '<rect x="0" y="0" width="200" height="200" fill="white"/>'
    '<circle cx="100" cy="100" r="60" fill="#4A90D9"/>'
    '</svg>'
)

# ── Number rounding ─────────────────────────────────────────────────────────────
# Training data has float32 artifacts: 93.30000305175781 → round to 1 decimal
_NUM_RE = re.compile(r'(-?\d+\.\d{2,})')

def _round_num(m):
    v = float(m.group(1))
    # If the value is very close to an integer, strip decimals
    if abs(v - round(v)) < 0.05:
        return str(int(round(v)))
    return f"{v:.1f}"


def optimize_svg(svg_text: str) -> str:
    """
    Post-process SVG for better score:
    1. Round numbers to 1 decimal (huge size reduction, helps compactness)
    2. Normalize viewBox 0.0 0.0 200.0 200.0 → 0 0 200 200
    3. Remove non-standard `filling` attribute (training data artifact)
    4. Strip XML comments
    5. Normalize whitespace inside attribute values (not between tags)
    """
    # Strip XML comments
    svg_text = re.sub(r'<!--.*?-->', '', svg_text, flags=re.DOTALL)

    # Remove non-standard filling attribute
    svg_text = re.sub(r'\s+filling=["\'][^"\']*["\']', '', svg_text)

    # Round all multi-decimal numbers
    svg_text = _NUM_RE.sub(_round_num, svg_text)

    # Normalize viewBox values (0.0 → 0)
    def _fix_viewbox(m):
        parts = m.group(1).split()
        normed = []
        for p in parts:
            try:
                f = float(p)
                normed.append(str(int(f)) if f == int(f) else f"{f:.1f}")
            except ValueError:
                normed.append(p)
        return f'viewBox="{" ".join(normed)}"'
    svg_text = re.sub(r'viewBox=["\']([^"\']+)["\']', _fix_viewbox, svg_text)

    # Collapse redundant whitespace inside d="..." path data
    # e.g. "M 10  20  L  30  40" → "M 10 20 L 30 40"
    def _compact_path(m):
        return 'd="' + re.sub(r'  +', ' ', m.group(1)).strip() + '"'
    svg_text = re.sub(r'd="([^"]+)"', _compact_path, svg_text)

    return svg_text


def fix_svg(svg_text: str) -> str:
    """
    Normalize SVG canvas:
    - Set width=256, height=256
    - PRESERVE model's natural viewBox
    - Default missing viewBox to 0 0 200 200 (82% of training data)
    """
    if len(svg_text) > 16000:
        svg_text = svg_text[:16000]
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    svg_text = re.sub(r'\bwidth=["\'][^"\']*["\']', 'width="256"', svg_text, count=1)
    svg_text = re.sub(r'\bheight=["\'][^"\']*["\']', 'height="256"', svg_text, count=1)
    if 'width=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg width="256"', 1)
    if 'height=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg height="256"', 1)
    if 'viewBox=' not in svg_text and 'viewbox=' not in svg_text.lower():
        svg_text = svg_text.replace("<svg", '<svg viewBox="0 0 200 200"', 1)
    return svg_text


def repair_svg(svg_text: str) -> str:
    """Close open tags on truncated SVG output."""
    open_tags = re.findall(r'<([a-zA-Z][a-zA-Z0-9]*)[\s>]', svg_text)
    self_closing = {'path','rect','circle','ellipse','line','polyline',
                    'polygon','stop','use','desc','title'}
    close_tags = re.findall(r'</([a-zA-Z][a-zA-Z0-9]*)>', svg_text)
    svg_text = re.sub(r'<[^>]*$', '', svg_text)
    stack = []
    for tag in open_tags:
        tl = tag.lower()
        if tl not in self_closing:
            stack.append(tl)
    for tag in close_tags:
        tl = tag.lower()
        if tl in stack:
            stack.remove(tl)
    for tag in reversed(stack):
        svg_text += f'</{tag}>'
    if not svg_text.rstrip().endswith('</svg>'):
        svg_text += '</svg>'
    return svg_text


def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or len(svg_text) > 16000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if not root.tag.lower().endswith("svg"):
        return False
    path_count = 0
    for elem in root.iter():
        local = (elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag).lower()
        if local in DISALLOWED_TAGS:
            return False
        if local not in ALLOWED_TAGS:
            return False
        for attr in elem.attrib:
            if attr.lower().startswith("on"):
                return False
        for attr_name in ("href", "{http://www.w3.org/1999/xlink}href"):
            val = elem.attrib.get(attr_name, "")
            if val.startswith("http") or val.startswith("//"):
                return False
        if local == "path":
            path_count += 1
    return path_count <= 256


def score_candidate(svg_text: str) -> float:
    """
    Heuristic quality score for candidate selection.
    Higher = better candidate.
    Based on visual richness: more distinct elements / colors = richer image.
    """
    if not is_valid_svg(svg_text):
        return -1.0
    if svg_text == FALLBACK_SVG:
        return 0.0

    score = 0.0
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return 0.0

    shape_tags = {'path', 'rect', 'circle', 'ellipse', 'line', 'polyline', 'polygon', 'text'}
    container_tags = {'g', 'defs', 'symbol', 'marker', 'pattern'}
    gradient_tags = {'lineargradient', 'radialgradient'}

    element_count = 0
    colors = set()
    has_gradients = False
    has_groups = False

    for elem in root.iter():
        local = (elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag).lower()
        if local in shape_tags:
            element_count += 1
            # collect colors
            for attr in ('fill', 'stroke'):
                v = elem.attrib.get(attr, '')
                if v and v not in ('none', 'transparent', ''):
                    colors.add(v.lower())
        elif local in gradient_tags:
            has_gradients = True
        elif local == 'g':
            has_groups = True

    # More elements = richer visual
    score += min(element_count, 30) * 0.5      # cap at 30, each worth 0.5
    score += min(len(colors), 10) * 0.3         # color variety
    score += 2.0 if has_gradients else 0.0      # gradients = more visual detail
    score += 0.5 if has_groups else 0.0         # groups = structured SVG

    # Penalize if suspiciously short (probably incomplete)
    if len(svg_text) < 200:
        score *= 0.3

    return score


def extract_and_validate_svg(raw_text: str) -> tuple:
    """Extract → optimize → fix → validate. Try repair if invalid."""
    m = SVG_REGEX.search(raw_text)
    if m:
        svg = fix_svg(optimize_svg(m.group(0).strip()))
        if is_valid_svg(svg):
            return svg, True
    m2 = SVG_OPEN.search(raw_text)
    if m2:
        svg = fix_svg(optimize_svg(repair_svg(m2.group(0).strip())))
        if is_valid_svg(svg):
            return svg, True
    return FALLBACK_SVG, False


print("SVG helpers loaded.")

# Quick optimize test
_t = '<svg viewBox="0.0 0.0 200.0 200.0"><path filling="0" d="M93.30000305175781 21.20000457763672 L93.30000305175781 80.4000015258789"/></svg>'
print("Before:", _t)
print("After: ", optimize_svg(_t))

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────────────
import unsloth  # noqa: F401 — must be first
from unsloth import FastLanguageModel
from peft import PeftModel

print(f"Loading base model from: {BASE_MODEL_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

print(f"Applying LoRA adapter from: {LORA_PATH}")
model = PeftModel.from_pretrained(model, LORA_PATH)

FastLanguageModel.for_inference(model)
print("Model ready.")

In [ ]:
# ── Generation ──────────────────────────────────────────────────────────────────
# System prompt: matches training + hints at 200x200 space (82% of training data)
SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Given a text description, output only valid SVG markup. "
    "Rules:\n"
    "- Root element must be <svg> with xmlns=\"http://www.w3.org/2000/svg\", "
    "width=\"256\", height=\"256\", and a viewBox matching the coordinate space used\n"
    "- Default coordinate space: viewBox=\"0 0 200 200\" (use unless the image "
    "naturally requires a different scale)\n"
    "- Use only these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter\n"
    "- No scripts, no animation, no event handlers, no external references\n"
    "- Maximum 16000 characters, maximum 256 path elements\n"
    "- Match the visual complexity of the description accurately\n"
    "- Output only the SVG code, nothing else"
)


def build_prompt(user_text: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def _generate_raw(prompt_text: str, max_tokens: int,
                  temperature: float = None) -> str:
    """Generate raw text. temperature=None → greedy decoding."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    do_sample = temperature is not None
    kwargs = dict(
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        kwargs["temperature"] = temperature
        kwargs["top_p"] = 0.9
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def generate_svg(prompt: str) -> tuple:
    """
    Multi-candidate generation with quality-based selection.

    Strategy:
    - Generate NUM_CANDIDATES outputs (greedy + light sampling + medium sampling)
    - Score each valid candidate by visual richness heuristic
    - Return highest-scoring valid candidate
    - Fallback: repair best greedy output

    Returns (svg, is_valid, attempts_used)
    """
    chat = build_prompt(prompt)
    candidates = []

    for i, temp in enumerate(CAND_TEMPS):
        raw = _generate_raw(chat, MAX_NEW_TOKENS, temperature=temp)
        svg, valid = extract_and_validate_svg(raw)
        s = score_candidate(svg)
        candidates.append((svg, valid, s, i + 1, raw))

        # Early exit: if first greedy candidate scores high, skip rest
        if i == 0 and valid and s >= 8.0:
            return svg, True, 1

    # Pick best valid candidate by score
    valid_cands = [(svg, sc, att) for svg, valid, sc, att, _ in candidates if valid]
    if valid_cands:
        best = max(valid_cands, key=lambda x: x[1])
        return best[0], True, best[2]

    # All invalid: return first (greedy) candidate (repaired)
    return candidates[0][0], False, len(CAND_TEMPS)


# Smoke test
test_svg, valid, attempts = generate_svg("a simple blue circle on white background")
print(f"Smoke test — valid: {valid} | attempts: {attempts} | score: {score_candidate(test_svg):.1f}")
print(test_svg[:400])

In [ ]:
# ── Inference with checkpointing ────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")

rows = []
done_ids = set()
if Path(CHECKPOINT_PATH).exists():
    ckpt = pd.read_csv(CHECKPOINT_PATH)
    rows = ckpt.to_dict('records')
    done_ids = set(ckpt['id'])
    print(f"Resuming from checkpoint — {len(done_ids)} already done")

fallback_count = 0
retry_count    = 0
score_sum      = 0.0
t0 = time.time()

for i, row in test_df.iterrows():
    if row['id'] in done_ids:
        continue

    svg, valid, attempts = generate_svg(str(row['prompt']))
    if not valid:
        fallback_count += 1
    if attempts > 1:
        retry_count += 1
    sc = score_candidate(svg)
    score_sum += max(sc, 0)
    rows.append({'id': row['id'], 'svg': svg})

    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, index=False)

    if (i + 1) % 10 == 0:
        elapsed = (time.time() - t0) / 60
        done = len(rows)
        eta = (elapsed / done) * (len(test_df) - done) if done else 0
        avg_sc = score_sum / done if done else 0
        print(f"  [{done}/{len(test_df)}] {elapsed:.1f}m | ETA {eta:.1f}m "
              f"| fallbacks: {fallback_count} | retries: {retry_count} | avg_score: {avg_sc:.1f}")

elapsed_total = (time.time() - t0) / 60
print(f"\nDone. {elapsed_total:.1f} min | Fallbacks: {fallback_count} | Retries: {retry_count}")

In [ ]:
# ── Save & validate submission ───────────────────────────────────────────────────
sub_df = pd.DataFrame(rows)

# Final validation pass — replace any invalid with fallback
invalid_mask = sub_df['svg'].apply(lambda s: not is_valid_svg(s))
print(f"Invalid SVGs before final fix: {invalid_mask.sum()}")
sub_df.loc[invalid_mask, 'svg'] = FALLBACK_SVG

# Size stats
sizes = sub_df['svg'].apply(len)
print(f"SVG size: min={sizes.min()} median={int(sizes.median())} max={sizes.max()} avg={int(sizes.mean())}")

sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}")
print(f"Shape: {sub_df.shape}")
print(f"Final invalid: {sub_df['svg'].apply(lambda s: not is_valid_svg(s)).sum()}")
print(sub_df.head(3))